<a href="https://colab.research.google.com/github/aiserhucui/news-recommendation-demo/blob/main/ASR_FINAL_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Complete Production-Ready Speech Recognition System
# With Hugging Face Datasets, Visualization, and 30-min Audio Support

# ============ PART 1: INSTALLATION & IMPORTS ============
!pip install torch torchaudio transformers datasets soundfile librosa gradio jiwer matplotlib seaborn scikit-learn wandb

import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from datasets import load_dataset, concatenate_datasets
import soundfile as sf
import librosa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import gradio as gr
from jiwer import wer, cer
from tqdm import tqdm
import json
import warnings
from pathlib import Path
import random
import string
import shutil
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from IPython.display import Audio as IPDAudio, display
import urllib.request
import tarfile
import zipfile
from sklearn.metrics import confusion_matrix, classification_report
from datetime import datetime
import time
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ============ PART 2: DATA PREPARATION - HUGGING FACE DATASETS ============

class HuggingFaceAudioDataset(Dataset):
    """
    Universal loader for Hugging Face audio datasets
    Handles multiple popular speech datasets without torchcodec
    """

    SUPPORTED_DATASETS = {
        'superb': {'config': 'asr', 'audio_col': 'audio', 'text_col': 'text'},
        'google/fleurs': {'config': 'en_us', 'audio_col': 'audio', 'text_col': 'transcription'},
        'facebook/voxpopuli': {'config': 'en', 'audio_col': 'audio', 'text_col': 'normalized_text'},
        'speechcolab/gigaspeech': {'config': 's', 'audio_col': 'audio', 'text_col': 'text'},
        'MLCommons/ml_spoken_words': {'config': 'en_wav', 'audio_col': 'audio', 'text_col': 'label'},
    }

    def __init__(self, dataset_name='google/fleurs', split='train', max_samples=1000, max_duration=30.0):
        self.max_duration = max_duration
        self.data = []

        print(f"Loading {dataset_name} from Hugging Face...")

        # Get dataset configuration
        if dataset_name in self.SUPPORTED_DATASETS:
            config = self.SUPPORTED_DATASETS[dataset_name]
            dataset_config = config['config']
            audio_column = config['audio_col']
            text_column = config['text_col']
        else:
            # Try generic approach
            dataset_config = None
            audio_column = 'audio'
            text_column = 'text'

        try:
            # Load dataset with streaming to avoid large downloads
            if dataset_config:
                dataset = load_dataset(dataset_name, dataset_config, split=split, streaming=True)
            else:
                dataset = load_dataset(dataset_name, split=split, streaming=True)

            # Process samples
            count = 0
            for item in tqdm(dataset, desc=f"Loading {dataset_name}", total=max_samples):
                if count >= max_samples:
                    break

                try:
                    # Extract audio and text
                    if audio_column in item and item[audio_column] is not None:
                        audio_data = item[audio_column]

                        # Handle different audio formats
                        if isinstance(audio_data, dict):
                            if 'array' in audio_data and 'sampling_rate' in audio_data:
                                audio_array = np.array(audio_data['array'], dtype=np.float32)
                                sr = audio_data['sampling_rate']
                            elif 'path' in audio_data:
                                # Load from path
                                audio_array, sr = librosa.load(audio_data['path'], sr=16000)
                            else:
                                continue
                        else:
                            continue

                        # Get text
                        text = str(item.get(text_column, '')).lower()

                        if len(audio_array) > 0 and text:
                            # Convert to tensor
                            waveform = torch.from_numpy(audio_array).float()
                            if waveform.dim() == 1:
                                waveform = waveform.unsqueeze(0)

                            # Check duration
                            duration = waveform.shape[1] / sr
                            if duration <= self.max_duration:
                                self.data.append({
                                    'waveform': waveform,
                                    'sample_rate': sr,
                                    'text': text
                                })
                                count += 1

                except Exception as e:
                    continue

            print(f"Successfully loaded {len(self.data)} samples from {dataset_name}")

        except Exception as e:
            print(f"Failed to load {dataset_name}: {e}")
            print("Creating synthetic fallback data...")
            self._create_synthetic_data(max_samples)

    def _create_synthetic_data(self, num_samples):
        """Create synthetic data for testing when real data unavailable"""

        texts = [
            "the quick brown fox jumps over the lazy dog",
            "artificial intelligence is transforming speech recognition",
            "deep learning models require large amounts of training data",
            "neural networks can learn complex patterns in audio signals",
            "speech recognition systems convert audio to text automatically"
        ]

        print(f"Creating {num_samples} synthetic samples...")

        for i in range(min(num_samples, 100)):
            # Generate synthetic audio
            duration = random.uniform(2.0, 5.0)
            sr = 16000
            t = np.linspace(0, duration, int(sr * duration))

            # Create more realistic synthetic speech-like audio
            fundamental = random.uniform(100, 200)
            harmonics = [1, 2, 3, 4, 5]
            signal = np.zeros_like(t)

            for h in harmonics:
                signal += (1.0/(h*h)) * np.sin(2 * np.pi * fundamental * h * t + random.random() * np.pi)

            # Add formants (speech-like resonances)
            for formant in [700, 1220, 2600]:
                signal += 0.1 * np.sin(2 * np.pi * formant * t)

            # Add noise and normalize
            signal += 0.02 * np.random.randn(len(t))
            signal = signal / (np.max(np.abs(signal)) + 1e-8) * 0.5

            waveform = torch.from_numpy(signal).float().unsqueeze(0)

            self.data.append({
                'waveform': waveform,
                'sample_rate': sr,
                'text': random.choice(texts)
            })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return item['waveform'], item['sample_rate'], item['text']

# ============ PART 3: MINI-LIBRISPEECH DATASET ============

def download_mini_librispeech():
    """Download mini-LibriSpeech dataset"""

    data_dir = Path('./speech_data')
    data_dir.mkdir(exist_ok=True)

    urls = {
        'train': 'http://www.openslr.org/resources/31/train-clean-5.tar.gz',
        'dev': 'http://www.openslr.org/resources/31/dev-clean-2.tar.gz'
    }

    for split, url in urls.items():
        tar_path = data_dir / f'{split}.tar.gz'
        extract_path = data_dir / 'LibriSpeech'

        if not extract_path.exists() or not list(extract_path.glob('*')):
            print(f"Downloading {split} dataset...")
            try:
                urllib.request.urlretrieve(url, tar_path)
                print(f"Extracting {split} dataset...")
                with tarfile.open(tar_path, 'r:gz') as tar:
                    tar.extractall(data_dir)
                tar_path.unlink()
                print(f"✓ {split} dataset ready")
            except Exception as e:
                print(f"Failed to download {split}: {e}")
                return None
        else:
            print(f"✓ {split} dataset already exists")

    return data_dir

class MiniLibriSpeechDataset(Dataset):
    """Mini-LibriSpeech dataset loader"""

    def __init__(self, data_dir, split='train'):
        self.data = []

        if split == 'train':
            search_dir = Path(data_dir) / 'LibriSpeech' / 'train-clean-5'
        else:
            search_dir = Path(data_dir) / 'LibriSpeech' / 'dev-clean-2'

        if not search_dir.exists():
            print(f"Dataset not found at {search_dir}")
            return

        print(f"Loading {split} data from {search_dir}...")

        for trans_file in search_dir.rglob('*.trans.txt'):
            transcripts = {}
            with open(trans_file, 'r') as f:
                for line in f:
                    parts = line.strip().split(' ', 1)
                    if len(parts) == 2:
                        transcripts[parts[0]] = parts[1].lower()

            for audio_file in trans_file.parent.glob('*.flac'):
                file_id = audio_file.stem
                if file_id in transcripts:
                    self.data.append({
                        'path': str(audio_file),
                        'text': transcripts[file_id]
                    })

        print(f"Loaded {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        waveform, sr = torchaudio.load(item['path'])
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        return waveform, sr, item['text']

# ============ PART 4: CUSTOM AUDIO UPLOAD HANDLER ============

class CustomAudioDataset(Dataset):
    """Handle user-uploaded audio including 30-minute files"""

    def __init__(self, chunk_duration=10.0):
        self.chunk_duration = chunk_duration
        self.target_sr = 16000
        self.data = []

    def add_audio_file(self, audio_path, transcript="", auto_chunk=True):
        """Add audio file with automatic chunking for long files"""

        try:
            print(f"Processing {Path(audio_path).name}...")
            waveform, sr = torchaudio.load(audio_path)

            # Convert to mono
            if waveform.shape[0] > 1:
                waveform = torch.mean(waveform, dim=0, keepdim=True)

            # Resample
            if sr != self.target_sr:
                resampler = T.Resample(sr, self.target_sr)
                waveform = resampler(waveform)

            duration = waveform.shape[1] / self.target_sr
            print(f"  Duration: {duration:.1f} seconds")

            if auto_chunk and duration > self.chunk_duration:
                # Chunk long audio
                num_chunks = int(np.ceil(duration / self.chunk_duration))
                chunk_samples = int(self.chunk_duration * self.target_sr)

                print(f"  Splitting into {num_chunks} chunks...")

                for i in range(num_chunks):
                    start = i * chunk_samples
                    end = min((i + 1) * chunk_samples, waveform.shape[1])
                    chunk = waveform[:, start:end]

                    # Pad if necessary
                    if chunk.shape[1] < chunk_samples:
                        chunk = F.pad(chunk, (0, chunk_samples - chunk.shape[1]))

                    self.data.append({
                        'waveform': chunk,
                        'text': transcript,
                        'filename': f"{Path(audio_path).stem}_chunk_{i}"
                    })
            else:
                self.data.append({
                    'waveform': waveform,
                    'text': transcript,
                    'filename': Path(audio_path).stem
                })

            return True

        except Exception as e:
            print(f"Error processing {audio_path}: {e}")
            return False

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return item['waveform'], self.target_sr, item['text']

# Upload helper for Colab
def upload_audio_files():
    """Upload audio files in Colab"""

    try:
        from google.colab import files

        print("="*60)
        print("UPLOAD AUDIO FILES (up to 30 minutes each)")
        print("="*60)
        print("Supported formats: WAV, MP3, FLAC, M4A, OGG")
        print("Files will be automatically chunked if >10 seconds")
        print("="*60)

        uploaded = files.upload()

        os.makedirs('/content/uploaded_audio', exist_ok=True)

        audio_files = []
        for filename, content in uploaded.items():
            if filename.lower().endswith(('.wav', '.mp3', '.flac', '.m4a', '.ogg')):
                path = f'/content/uploaded_audio/{filename}'
                with open(path, 'wb') as f:
                    f.write(content)
                audio_files.append(path)
                print(f"✓ Uploaded: {filename}")

        return audio_files

    except ImportError:
        print("Not in Colab. Place files in /content/uploaded_audio/")
        return []

# ============ PART 5: FEATURE ENGINEERING & VISUALIZATION ============

class AudioFeatureExtractor:
    """Comprehensive audio feature extraction with visualization"""

    def __init__(self, sample_rate=16000, n_mfcc=13, n_mels=80):
        self.sample_rate = sample_rate
        self.n_mfcc = n_mfcc
        self.n_mels = n_mels

        # Feature extractors
        self.mfcc = T.MFCC(
            sample_rate=sample_rate,
            n_mfcc=n_mfcc,
            melkwargs={'n_fft': 400, 'hop_length': 160, 'n_mels': n_mels}
        )

        self.melspec = T.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=400,
            hop_length=160,
            n_mels=n_mels
        )

    def extract_mfcc(self, waveform):
        """Extract MFCC with deltas"""

        if waveform.dim() == 1:
            waveform = waveform.unsqueeze(0)

        # Extract MFCCs
        mfcc = self.mfcc(waveform)

        # Add delta and delta-delta
        delta1 = torchaudio.functional.compute_deltas(mfcc)
        delta2 = torchaudio.functional.compute_deltas(delta1)

        # Stack features
        features = torch.cat([mfcc, delta1, delta2], dim=1)

        return features

    def visualize_features(self, waveform, sr, text="", save_path=None):
        """Visualize audio features"""

        if waveform.dim() == 1:
            waveform = waveform.unsqueeze(0)

        # Extract features
        mfcc = self.extract_mfcc(waveform)
        melspec = self.melspec(waveform)
        melspec_db = librosa.power_to_db(melspec.squeeze().numpy())

        # Create figure
        fig, axes = plt.subplots(4, 1, figsize=(14, 12))

        # 1. Waveform
        time_axis = np.linspace(0, waveform.shape[1]/sr, waveform.shape[1])
        axes[0].plot(time_axis, waveform.squeeze().numpy(), linewidth=0.5)
        axes[0].set_title(f'Waveform: "{text[:50]}..."' if len(text) > 50 else f'Waveform: "{text}"')
        axes[0].set_xlabel('Time (s)')
        axes[0].set_ylabel('Amplitude')
        axes[0].grid(True, alpha=0.3)

        # 2. Mel Spectrogram
        img1 = axes[1].imshow(melspec_db, aspect='auto', origin='lower',
                              cmap='viridis', extent=[0, waveform.shape[1]/sr, 0, self.n_mels])
        axes[1].set_title('Mel Spectrogram')
        axes[1].set_xlabel('Time (s)')
        axes[1].set_ylabel('Mel Frequency Bins')
        plt.colorbar(img1, ax=axes[1], format='%+2.0f dB')

        # 3. MFCC
        img2 = axes[2].imshow(mfcc.squeeze()[:self.n_mfcc].numpy(), aspect='auto',
                              origin='lower', cmap='coolwarm')
        axes[2].set_title('MFCC Features')
        axes[2].set_xlabel('Time (frames)')
        axes[2].set_ylabel('MFCC Coefficients')
        plt.colorbar(img2, ax=axes[2])

        # 4. MFCC Statistics
        mfcc_mean = mfcc.squeeze()[:self.n_mfcc].mean(dim=1).numpy()
        mfcc_std = mfcc.squeeze()[:self.n_mfcc].std(dim=1).numpy()
        x_pos = np.arange(self.n_mfcc)

        axes[3].bar(x_pos, mfcc_mean, yerr=mfcc_std, capsize=5, alpha=0.7)
        axes[3].set_title('MFCC Statistics (Mean ± Std)')
        axes[3].set_xlabel('MFCC Coefficient')
        axes[3].set_ylabel('Value')
        axes[3].grid(True, alpha=0.3)

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=100, bbox_inches='tight')

        plt.show()

        return mfcc

# ============ PART 6: TEXT PROCESSING & VOCABULARY ============

class TextVocabulary:
    """Character-level vocabulary with statistics"""

    def __init__(self):
        # Comprehensive character set
        chars = list(" abcdefghijklmnopqrstuvwxyz0123456789,.?!'\"()-")
        special = ['<pad>', '<sos>', '<eos>', '<blank>']

        self.chars = special + chars
        self.char2idx = {ch: i for i, ch in enumerate(self.chars)}
        self.idx2char = {i: ch for i, ch in enumerate(self.chars)}
        self.blank_id = self.char2idx['<blank>']
        self.pad_id = self.char2idx['<pad>']
        self.vocab_size = len(self.chars)

    def encode(self, text):
        """Convert text to indices"""
        indices = []
        for ch in text.lower():
            if ch in self.char2idx:
                indices.append(self.char2idx[ch])
            elif ch.isspace():
                indices.append(self.char2idx[' '])

        return torch.tensor(indices if indices else [self.blank_id], dtype=torch.long)

    def decode(self, indices, remove_special=True):
        """Convert indices to text"""
        if isinstance(indices, torch.Tensor):
            indices = indices.cpu().numpy()

        chars = []
        special_tokens = {'<pad>', '<sos>', '<eos>', '<blank>'}

        for idx in indices:
            if idx < len(self.idx2char):
                ch = self.idx2char[idx]
                if not remove_special or ch not in special_tokens:
                    chars.append(ch)

        return ''.join(chars)

    def analyze_text_corpus(self, texts):
        """Analyze text statistics"""

        # Character frequency
        char_counts = Counter()
        word_lengths = []

        for text in texts:
            text = text.lower()
            char_counts.update(text)
            word_lengths.extend([len(word) for word in text.split()])

        # Visualize
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Character frequency
        chars, counts = zip(*char_counts.most_common(20))
        axes[0].bar(range(len(chars)), counts)
        axes[0].set_xticks(range(len(chars)))
        axes[0].set_xticklabels(chars)
        axes[0].set_title('Top 20 Character Frequencies')
        axes[0].set_xlabel('Character')
        axes[0].set_ylabel('Count')

        # Word length distribution
        axes[1].hist(word_lengths, bins=20, edgecolor='black', alpha=0.7)
        axes[1].set_title('Word Length Distribution')
        axes[1].set_xlabel('Word Length')
        axes[1].set_ylabel('Frequency')
        axes[1].axvline(np.mean(word_lengths), color='red', linestyle='--',
                       label=f'Mean: {np.mean(word_lengths):.1f}')
        axes[1].legend()

        plt.tight_layout()
        plt.show()

        return char_counts

# ============ PART 7: MODEL ARCHITECTURE ============

class ConvBlock(nn.Module):
    """Convolutional block with residual connection"""

    def __init__(self, in_channels, out_channels, kernel_size=3):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=kernel_size//2)
        self.batch_norm1 = nn.BatchNorm1d(out_channels)
        self.batch_norm2 = nn.BatchNorm1d(out_channels)
        self.dropout = nn.Dropout(0.1)

        # Residual connection
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        residual = self.residual(x)

        x = F.relu(self.batch_norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.batch_norm2(self.conv2(x))

        return F.relu(x + residual)

class ProductionASRModel(nn.Module):
    """Production-ready ASR model with advanced architecture"""

    def __init__(self, input_dim=39, hidden_dim=256, num_layers=6, vocab_size=50, dropout=0.2):
        super().__init__()

        # CNN encoder
        self.conv_blocks = nn.ModuleList([
            ConvBlock(input_dim, 128),
            ConvBlock(128, 256),
            ConvBlock(256, 256),
        ])

        self.pool = nn.MaxPool1d(2)

        # Calculate time reduction factor
        self.time_reduction_factor = 2  # Due to single MaxPool1d(2)

        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size=256,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        # Attention mechanism
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim * 2,
            num_heads=8,
            dropout=dropout,
            batch_first=True
        )

        # Output layers
        self.output_proj = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, vocab_size)
        )

    def forward(self, x):
        # x: (batch, features, time)

        # CNN encoding
        for conv_block in self.conv_blocks[:-1]:
            x = conv_block(x)

        # Apply pooling after second conv block
        x = self.pool(x)

        # Last conv block
        x = self.conv_blocks[-1](x)

        # Prepare for RNN: (batch, time, features)
        x = x.transpose(1, 2)

        # LSTM
        x, _ = self.lstm(x)

        # Self-attention
        x, _ = self.attention(x, x, x)

        # Output projection
        x = self.output_proj(x)

        return F.log_softmax(x, dim=-1)

    def get_model_size(self):
        """Get model size in MB"""
        param_size = sum(p.numel() * p.element_size() for p in self.parameters())
        buffer_size = sum(b.numel() * b.element_size() for b in self.buffers())
        size_mb = (param_size + buffer_size) / 1024 / 1024
        return size_mb

# ============ PART 8: DATASET WRAPPER ============

class ASRDataset(Dataset):
    """Wrapper for ASR datasets with augmentation"""

    def __init__(self, base_dataset, feature_extractor, vocab, max_len=500, augment=False):
        self.base_dataset = base_dataset
        self.feature_extractor = feature_extractor
        self.vocab = vocab
        self.max_len = max_len
        self.augment = augment

    def __len__(self):
        return len(self.base_dataset)

    def augment_audio(self, waveform, sr):
        """Apply audio augmentation"""

        if random.random() < 0.3:
            # Time stretching using torchaudio
            rate = random.uniform(0.9, 1.1)
            if rate != 1.0:
                resampler = T.Resample(sr, int(sr * rate))
                waveform = resampler(waveform)
                # Resample back to original rate
                resampler_back = T.Resample(int(sr * rate), sr)
                waveform = resampler_back(waveform)

        if random.random() < 0.3:
            # Add noise
            noise = torch.randn_like(waveform) * 0.005
            waveform = waveform + noise

        if random.random() < 0.3:
            # Volume change
            volume = random.uniform(0.8, 1.2)
            waveform = waveform * volume

        return waveform

    def __getitem__(self, idx):
        waveform, sr, text = self.base_dataset[idx]

        # Resample if needed
        if sr != 16000:
            resampler = T.Resample(sr, 16000)
            waveform = resampler(waveform)
            sr = 16000

        # Apply augmentation
        if self.augment:
            waveform = self.augment_audio(waveform, sr)

        # Extract features
        features = self.feature_extractor.extract_mfcc(waveform)

        # Encode text
        text_encoded = self.vocab.encode(text)

        # Truncate/pad features
        if features.shape[2] > self.max_len:
            features = features[:, :, :self.max_len]
        else:
            features = F.pad(features, (0, self.max_len - features.shape[2]))

        return features.squeeze(0), text_encoded, len(text_encoded)

def collate_fn(batch):
    """Custom collate function"""

    features = []
    targets = []
    feat_lens = []
    target_lens = []

    for feat, target, target_len in batch:
        features.append(feat)
        targets.append(target)
        feat_lens.append(feat.shape[1])
        target_lens.append(target_len)

    # Stack features
    features = torch.stack(features)

    # Pad targets
    max_target_len = max(target_lens) if target_lens else 1
    padded_targets = []

    for target in targets:
        if len(target) < max_target_len:
            padded = F.pad(target, (0, max_target_len - len(target)))
        else:
            padded = target[:max_target_len]
        padded_targets.append(padded)

    targets = torch.stack(padded_targets)

    return features, targets, torch.tensor(feat_lens), torch.tensor(target_lens)

# ============ PART 9: TRAINING & EVALUATION ============

class Trainer:
    """Comprehensive training with metrics and visualization"""

    def __init__(self, model, train_loader, val_loader, vocab, device='cpu', learning_rate=1e-3):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.vocab = vocab
        self.device = device

        # Optimizer and scheduler
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', factor=0.5, patience=3
        )

        # Loss function
        self.ctc_loss = nn.CTCLoss(blank=vocab.blank_id, zero_infinity=True)

        # Metrics storage
        self.train_losses = []
        self.val_losses = []
        self.val_wers = []
        self.val_cers = []
        self.learning_rates = []

    def train_epoch(self):
        """Train for one epoch"""

        self.model.train()
        total_loss = 0

        progress_bar = tqdm(self.train_loader, desc="Training")
        for batch in progress_bar:
            features, targets, feat_lens, target_lens = batch
            features = features.to(self.device)
            targets = targets.to(self.device)

            self.optimizer.zero_grad()

            # Forward pass
            output = self.model(features)
            output = output.transpose(0, 1)  # (time, batch, vocab)

            # Adjust feature lengths for CNN downsampling
            adjusted_feat_lens = feat_lens // self.model.time_reduction_factor

            # Compute loss
            loss = self.ctc_loss(output, targets, adjusted_feat_lens, target_lens)

            # Backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.optimizer.step()

            total_loss += loss.item()
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

        return total_loss / len(self.train_loader)

    def evaluate(self):
        """Evaluate on validation set"""

        self.model.eval()
        total_loss = 0
        all_predictions = []
        all_targets = []

        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc="Validation"):
                features, targets, feat_lens, target_lens = batch
                features = features.to(self.device)
                targets = targets.to(self.device)

                # Forward pass
                output = self.model(features)
                output_t = output.transpose(0, 1)

                # Adjust feature lengths
                adjusted_feat_lens = feat_lens // self.model.time_reduction_factor

                # Compute loss
                loss = self.ctc_loss(output_t, targets, adjusted_feat_lens, target_lens)
                total_loss += loss.item()

                # Decode predictions
                predictions = self.greedy_decode(output)

                # Collect for metrics
                for i in range(len(predictions)):
                    pred_text = self.vocab.decode(predictions[i])
                    target_text = self.vocab.decode(targets[i][:target_lens[i]])
                    all_predictions.append(pred_text)
                    all_targets.append(target_text)

        # Calculate metrics
        avg_loss = total_loss / len(self.val_loader)
        wer_score = wer(all_targets, all_predictions) if all_targets else 1.0
        cer_score = cer(all_targets, all_predictions) if all_targets else 1.0

        # Sample predictions for display
        samples = list(zip(all_targets[:5], all_predictions[:5]))

        return avg_loss, wer_score, cer_score, samples

    def greedy_decode(self, output):
        """Greedy CTC decoding"""

        argmax = output.argmax(dim=-1)
        decoded = []

        for sequence in argmax:
            chars = []
            prev = -1

            for idx in sequence:
                if idx != self.vocab.blank_id and idx != prev:
                    chars.append(idx.item())
                prev = idx

            decoded.append(chars)

        return decoded

    def train(self, num_epochs=10):
        """Complete training loop"""

        print(f"\n{'='*60}")
        print(f"TRAINING FOR {num_epochs} EPOCHS")
        print(f"{'='*60}")

        best_wer = float('inf')

        for epoch in range(num_epochs):
            start_time = time.time()

            # Training
            train_loss = self.train_epoch()

            # Validation
            val_loss, wer_score, cer_score, samples = self.evaluate()

            # Store metrics
            self.train_losses.append(train_loss)
            self.val_losses.append(val_loss)
            self.val_wers.append(wer_score)
            self.val_cers.append(cer_score)
            self.learning_rates.append(self.optimizer.param_groups[0]['lr'])

            # Update scheduler
            self.scheduler.step(val_loss)

            # Print epoch summary
            epoch_time = time.time() - start_time
            print(f"\nEpoch {epoch+1}/{num_epochs} - Time: {epoch_time:.1f}s")
            print(f"  Train Loss: {train_loss:.4f}")
            print(f"  Val Loss: {val_loss:.4f}")
            print(f"  WER: {wer_score:.2%}")
            print(f"  CER: {cer_score:.2%}")
            print(f"  LR: {self.optimizer.param_groups[0]['lr']:.2e}")

            # Print sample predictions
            print("\n  Sample Predictions:")
            for target, pred in samples[:3]:
                print(f"    Target: {target}")
                print(f"    Pred:   {pred}")
                print()

            # Save best model
            if wer_score < best_wer:
                best_wer = wer_score
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'wer': wer_score,
                    'cer': cer_score,
                    'vocab': self.vocab
                }, 'best_asr_model.pth')
                print(f"  ✓ Saved best model (WER: {wer_score:.2%})")

    def plot_training_history(self):
        """Plot training metrics"""

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        # Loss curves
        axes[0, 0].plot(self.train_losses, label='Train Loss', linewidth=2)
        axes[0, 0].plot(self.val_losses, label='Val Loss', linewidth=2)
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].set_title('Training and Validation Loss')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)

        # WER curve
        axes[0, 1].plot(self.val_wers, 'r-', linewidth=2)
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('WER')
        axes[0, 1].set_title('Word Error Rate')
        axes[0, 1].grid(True, alpha=0.3)
        axes[0, 1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.0%}'.format(y)))

        # CER curve
        axes[1, 0].plot(self.val_cers, 'g-', linewidth=2)
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('CER')
        axes[1, 0].set_title('Character Error Rate')
        axes[1, 0].grid(True, alpha=0.3)
        axes[1, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.0%}'.format(y)))

        # Learning rate
        axes[1, 1].plot(self.learning_rates, 'b-', linewidth=2)
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Learning Rate')
        axes[1, 1].set_title('Learning Rate Schedule')
        axes[1, 1].grid(True, alpha=0.3)
        axes[1, 1].set_yscale('log')

        plt.tight_layout()
        plt.show()

# ============ PART 10: MAIN EXECUTION ============

print("\n" + "="*60)
print("COMPREHENSIVE SPEECH RECOGNITION SYSTEM")
print("="*60)

# 1. Data Loading
print("\n1. LOADING DATASETS...")
print("-" * 40)

datasets_loaded = []

# Try Mini-LibriSpeech
try:
    data_dir = download_mini_librispeech()
    if data_dir:
        train_mini = MiniLibriSpeechDataset(data_dir, 'train')
        val_mini = MiniLibriSpeechDataset(data_dir, 'dev')
        if len(train_mini) > 0:
            datasets_loaded.append(('Mini-LibriSpeech', train_mini, val_mini))
            print(f"✓ Mini-LibriSpeech: {len(train_mini)} train, {len(val_mini)} val")
except Exception as e:
    print(f"✗ Mini-LibriSpeech failed: {e}")

# Try Hugging Face datasets
hf_configs = [
    ('google/fleurs', 'en_us', 500),
    ('facebook/voxpopuli', 'en', 300),
    ('superb', None, 200)
]

for dataset_name, config, max_samples in hf_configs:
    try:
        print(f"Loading {dataset_name}...")
        if config:
            train_hf = HuggingFaceAudioDataset(dataset_name, 'train', max_samples)
        else:
            train_hf = HuggingFaceAudioDataset(dataset_name, 'train', max_samples)

        if len(train_hf) > 50:  # Only use if we got reasonable amount of data
            val_hf = HuggingFaceAudioDataset(dataset_name, 'validation', max_samples//5)
            datasets_loaded.append((dataset_name, train_hf, val_hf))
            print(f"✓ {dataset_name}: {len(train_hf)} train, {len(val_hf)} val")
            break  # Use first successful
    except Exception as e:
        print(f"✗ {dataset_name} failed: {e}")

# Combine datasets
if datasets_loaded:
    if len(datasets_loaded) == 1:
        dataset_name, train_dataset, val_dataset = datasets_loaded[0]
    else:
        # Combine all
        train_datasets = [d[1] for d in datasets_loaded]
        val_datasets = [d[2] for d in datasets_loaded]
        train_dataset = ConcatDataset(train_datasets)
        val_dataset = ConcatDataset(val_datasets)
        dataset_name = "Combined"
else:
    # Fallback to synthetic
    print("Using synthetic data...")
    train_dataset = HuggingFaceAudioDataset('synthetic', 'train', 500)
    val_dataset = HuggingFaceAudioDataset('synthetic', 'val', 100)
    dataset_name = "Synthetic"

print(f"\n✓ Final dataset: {dataset_name}")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Val: {len(val_dataset)} samples")

# 2. Check for custom uploads
custom_dataset = CustomAudioDataset()
if os.path.exists('/content/uploaded_audio'):
    audio_files = list(Path('/content/uploaded_audio').glob('*.*'))
    if audio_files:
        print(f"\n✓ Found {len(audio_files)} uploaded files")
        for audio_file in audio_files[:5]:  # Process first 5
            custom_dataset.add_audio_file(str(audio_file))

        if len(custom_dataset) > 0:
            train_dataset = ConcatDataset([train_dataset, custom_dataset])
            print(f"  Added {len(custom_dataset)} custom samples")

# 3. Feature Extraction Setup
print("\n2. FEATURE EXTRACTION...")
print("-" * 40)

feature_extractor = AudioFeatureExtractor()
vocab = TextVocabulary()

print(f"✓ Feature extractor initialized")
print(f"✓ Vocabulary size: {vocab.vocab_size}")

# 4. Visualize sample
print("\n3. VISUALIZING SAMPLE...")
print("-" * 40)

sample_waveform, sample_sr, sample_text = train_dataset[0]
_ = feature_extractor.visualize_features(sample_waveform, sample_sr, sample_text)

# 5. Create datasets
print("\n4. PREPARING DATA LOADERS...")
print("-" * 40)

train_wrapped = ASRDataset(train_dataset, feature_extractor, vocab, augment=True)
val_wrapped = ASRDataset(val_dataset, feature_extractor, vocab, augment=False)

batch_size = 16 if device.type == 'cuda' else 8

train_loader = DataLoader(
    train_wrapped,
    batch_size=min(batch_size, len(train_wrapped)),
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2 if device.type == 'cuda' else 0
)

val_loader = DataLoader(
    val_wrapped,
    batch_size=min(batch_size, len(val_wrapped)),
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2 if device.type == 'cuda' else 0
)

print(f"✓ Train batches: {len(train_loader)}")
print(f"✓ Val batches: {len(val_loader)}")

# 6. Model initialization
print("\n5. INITIALIZING MODEL...")
print("-" * 40)

model = ProductionASRModel(
    input_dim=39,  # 13 MFCC * 3
    hidden_dim=256,
    num_layers=4,
    vocab_size=vocab.vocab_size,
    dropout=0.2
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✓ Model initialized")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size: {model.get_model_size():.2f} MB")

# 7. Training
print("\n6. TRAINING...")
print("-" * 40)

trainer = Trainer(model, train_loader, val_loader, vocab, device)

# Train for fewer epochs in demo, increase for production
NUM_EPOCHS = 5  # Increase to 20-30 for production

trainer.train(NUM_EPOCHS)

# 8. Plot metrics
print("\n7. PLOTTING METRICS...")
print("-" * 40)

trainer.plot_training_history()

# ============ PART 11: INFERENCE & GRADIO INTERFACE ============

def transcribe_audio(audio_file, chunk_duration=10.0):
    """Transcribe audio file with chunking for long files"""

    try:
        model.eval()

        # Load audio
        waveform, sr = torchaudio.load(audio_file)

        # Convert to mono
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Resample
        if sr != 16000:
            resampler = T.Resample(sr, 16000)
            waveform = resampler(waveform)
            sr = 16000

        duration = waveform.shape[1] / sr

        # Process in chunks
        chunk_samples = int(chunk_duration * sr)
        transcriptions = []

        for start in range(0, waveform.shape[1], chunk_samples):
            end = min(start + chunk_samples, waveform.shape[1])
            chunk = waveform[:, start:end]

            # Pad if needed
            if chunk.shape[1] < chunk_samples:
                chunk = F.pad(chunk, (0, chunk_samples - chunk.shape[1]))

            # Extract features
            features = feature_extractor.extract_mfcc(chunk)

            # Pad features
            max_len = 500
            if features.shape[2] > max_len:
                features = features[:, :, :max_len]
            else:
                features = F.pad(features, (0, max_len - features.shape[2]))

            features = features.unsqueeze(0).to(device)

            # Predict
            with torch.no_grad():
                output = model(features)

                # Decode
                pred = trainer.greedy_decode(output)[0]
                text = vocab.decode(pred)

                if text.strip():
                    transcriptions.append(text)

        full_transcription = ' '.join(transcriptions)

        # Return with metadata
        metadata = f"Duration: {duration:.1f}s | Chunks: {len(transcriptions)}"

        return full_transcription if full_transcription else "No speech detected", metadata

    except Exception as e:
        return f"Error: {str(e)}", "Processing failed"

# Create Gradio interface
print("\n8. LAUNCHING GRADIO INTERFACE...")
print("-" * 40)

iface = gr.Interface(
    fn=transcribe_audio,
    inputs=gr.Audio(
        sources=["microphone", "upload"],
        type="filepath",
        label="Upload or Record Audio (up to 30 minutes)"
    ),
    outputs=[
        gr.Textbox(label="Transcription", lines=5),
        gr.Textbox(label="Processing Info")
    ],
    title="🎙️ Production Speech Recognition System",
    description=f"""
    **Model**: {dataset_name} dataset | {total_params:,} parameters

    **Features**:
    - Handles audio files up to 30 minutes
    - Automatic chunking for long recordings
    - Character-level CTC transcription
    - Trained on multiple datasets

    **Performance**: WER: {trainer.val_wers[-1]:.1%} | CER: {trainer.val_cers[-1]:.1%}

    **Instructions**:
    1. Upload an audio file or record using microphone
    2. Click Submit to transcribe
    3. Long files are automatically processed in 10-second chunks
    """,
    examples=None,
    theme="default",
    allow_flagging="manual"
)

iface.launch(debug=True, share=True)

# ============ FINAL MESSAGE ============
print("\n" + "="*60)
print("SYSTEM READY!")
print("="*60)
print("\nTo upload custom training audio:")
print("  audio_files = upload_audio_files()")
print("\nModel saved as: best_asr_model.pth")
print("="*60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 27.6 MB/s eta 0:00:00
Using device: cuda
PyTorch version: 2.11.0+cu128
CUDA available: True
CUDA version: 12.8
GPU: Tesla T4

COMPREHENSIVE SPEECH RECOGNITION SYSTEM

1. LOADING DATASETS...
----------------------------------------
Extracting train dataset...
✓ train dataset ready
✓ dev dataset already exists
Loading train data from speech_data/LibriSpeech/train-clean-5...
Loaded 1519 samples
Dataset not found at speech_data/LibriSpeech/dev-clean-2
✓ Mini-LibriSpeech: 1519 train, 0 val
Loading google/fleurs...
Loading google/fleurs from Hugging Face...


README.md:   0%|          | 0.00/386k [00:00<?, ?B/s]

Loading google/fleurs: 2602it [00:29, 88.08it/s] 


Successfully loaded 0 samples from google/fleurs
Loading facebook/voxpopuli...
Loading facebook/voxpopuli from Hugging Face...


README.md:   0%|          | 0.00/31.9k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading facebook/voxpopuli: 10080it [01:31, 416.52it/s]